In [ ]:
#! /usr/bin/env python

import rospy 
import actionlib.msg
import actionlib 
import sys

from geometry_msgs.msg import PoseStamped
from nav_msgs.msg import Odometry

import assignment_2_2024.msg
from assignment2_ros1.msg import RobotInfo

import ipywidgets as widgets
from IPython.display import display
import os
import platform

%matplotlib widget

import matplotlib.pyplot as plt
import tf

from tf.transformations import quaternion_matrix 
import numpy as np
from matplotlib.animation import FuncAnimation

In [ ]:
class Robot:
    def __init__(self):
        self.status_pub = rospy.Publisher("/robot_status", RobotInfo, queue_size=10)
        self.subscriber = rospy.Subscriber("/odom", Odometry, self.publish_robot_info)
        self.client = actionlib.SimpleActionClient('reaching_goal', assignment_2_2024.msg.PlanningAction)
        self.client.wait_for_server()
        
        self.target_x = 10
        self.target_y = 10
    
        self.latest_feedback = None
        
    def feedback_callback(self, feedback):
        global latest_feedback
        self.latest_feedback = feedback
        
    # this function send the coordinates set
    def sending_goal(self, x, y):
        # x, y are the sliders
        
        goal = assignment_2_2024.msg.PlanningGoal()   
        goal.target_pose = PoseStamped()
        goal.target_pose.pose.position.x = x.value
        goal.target_pose.pose.position.y = y.value

        self.client.send_goal(goal, feedback_cb = self.feedback_callback)

        with self.output:
            rospy.loginfo(f"Goal set and sent ({x.value}, {y.value})")
    
    def publish_robot_info(self, msg):
        robot_info = RobotInfo()
        robot_info.x = msg.pose.pose.position.x
        robot_info.y = msg.pose.pose.position.y
        robot_info.vel_x = msg.twist.twist.linear.x
        robot_info.vel_z = msg.twist.twist.angular.z

        self.status_pub.publish(robot_info)
        
    def cancel_goal(self):
        with self.output:
            if self.client.get_state() not in [actionlib.GoalStatus.SUCCEEDED, actionlib.GoalStatus.ABORTED, actionlib.GoalStatus.PREEMPTED]:
                rospy.loginfo("Cancelling goal")
                self.client.cancel_goal()
            else:
                rospy.loginfo("Goal already reached")
        
    
    def feedback(self):
        with self.output:
            rospy.loginfo("Getting feedback...")
            if self.latest_feedback is None:
                rospy.loginfo("Feedback still not received")
            else:
                rospy.loginfo("Feedback received: %s", self.latest_feedback)
        
    
    def exit(self):
        with self.output:
            print("Exit")

In [ ]:
class RobotGUI:
    def __init__(self, robot):
        self.robot = robot
        self.robot.output = widgets.Output()  # Create output widget
        
        self.cancel_goal_button = Button("Cancel goal", robot.cancel_goal)
        self.feedback_button = Button("Feedback", robot.feedback)
        self.clear_button = Button("Clear console", self.clear_console)
        
        self.slider_x = widgets.FloatSlider(min=-10, max=10) 
        self.slider_y = widgets.FloatSlider(min=-10, max=10) 
        self.coordinate_button = Button("Set coordinates", lambda: robot.sending_goal(self.slider_x, self.slider_y))

        left_column = widgets.VBox([
            self.cancel_goal_button.object,
            self.feedback_button.object,
            
            self.slider_x,
            self.slider_y,
            self.coordinate_button.object,
            self.clear_button.object
        ])

        # Layout finale con due colonne
        layout = widgets.HBox([
            left_column,
            self.robot.output
        ])
        
        display(layout)
    
    def display_coordinates_slider(self):
        display(self.slider_x)
        display(self.slider_y)
    
    def clear_console(self):
        self.robot.output.clear_output()

class Button:
    def __init__(self, label, command):
        self.label = label
        self.object = widgets.Button(description=self.label)
        self.command = command
        
        self.object.on_click(self.click)
        
    def click(self, _):
        self.command()
        
    def show(self):
        display(self.object)


In [ ]:
class Visualiser:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ln, = plt.plot([], [], 'k')
        self.x_data, self.y_data = [], []

    def plot_init(self):
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        return self.ln,
        
    def odom_callback(self, msg):
        self.y_data.append(msg.pose.pose.position.y)
        self.x_data.append(msg.pose.pose.position.x)

    def update_plot(self, frame):
        self.ln.set_data(self.x_data, self.y_data)
        return self.ln,
 


In [ ]:
class TargetVisualizer:
    def __init__(self, robot):
        self.robot = robot
        self.fig, (self.ax, self.ax_pie) = plt.subplots(1, 2, figsize=(10, 5))
        
        # Cartesian graph
        self.reached_ln, = self.ax.plot([], [], 'bo', label = "Reached")
        self.not_reached_ln, = self.ax.plot([], [], 'ro', label = "Not reached")
        self.x_reached, self.y_reached = [], []
        self.x_not_reached, self.y_not_reached = [], []
        
        self.last_x = None
        self.last_y = None
        
        # Pie graph
        self.pie = None
        

    def plot_init(self):
        # Cartesian graph
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        self.ax.set_xlabel("x axis")
        self.ax.set_ylabel("y axis")
        self.ax.legend()
        
        # Pie graph
        self.pie = self.ax_pie.pie([1], labels=['No Data'], autopct='%1.1f%%')
        self.ax_pie.set_title("Goal Status")
        
        return self.reached_ln, self.not_reached_ln
    
       
    def new_point(self, msg):
        self.last_x = msg.pose.pose.position.x
        self.last_y = msg.pose.pose.position.y
        current_x = float(rospy.get_param("/des_pos_x"))
        current_y = float(rospy.get_param("/des_pos_y"))
        if (self.robot.client.get_state() is not actionlib.GoalStatus.SUCCEEDED and 
            current_x not in self.x_not_reached 
            and current_y not in self.y_not_reached):
            
            #print(self.robot.client.get_state())
            self.x_not_reached.append(current_x)
            self.y_not_reached.append(current_y)
        if (self.robot.client.get_state() is actionlib.GoalStatus.SUCCEEDED and 
           current_x not in self.x_reached and
           current_y not in self.y_reached):
            self.x_not_reached.pop()
            self.y_not_reached.pop()
            self.x_reached.append(current_x)
            self.y_reached.append(current_y)
        
    
    def update_plot(self, frame):
        # Cartesian graph
        self.reached_ln.set_data(self.x_reached, self.y_reached)
        self.not_reached_ln.set_data(self.x_not_reached, self.y_not_reached)
        
        # Pie graph
        self.ax_pie.clear()
        tot = len(self.x_reached) + len(self.x_not_reached)
        if (tot > 0):
            labels = 'Reached', 'Not reached'
            sizes = [len(self.x_reached)/tot * 100, len(self.x_not_reached)/tot * 100]
        else: 
            labels = 'No data'
            sizes = [1]
        
        explode = (0.1, 0.1)
        self.ax_pie.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%', shadow=True, startangle=90)
        self.ax_pie.axis('equal')
        self.ax_pie.set_title("Goal Status")
        
        return self.reached_ln, self.not_reached_ln

In [ ]:
if __name__ == '__main__':
    try:
        # Initializes a rospy node
        rospy.init_node("action_client_node")
        rate = rospy.Rate(10)
        
        robot = Robot()
        robotGui = RobotGUI(robot)
    except rospy.ROSInterruptException:
        print("Action client interrupted", file=sys.stderr)        

In [ ]:
if __name__ == '__main__':
    try:
        vis = Visualiser()
        
        robot_position_sub = rospy.Subscriber('/odom', Odometry, vis.odom_callback)
        ani = FuncAnimation(vis.fig, vis.update_plot, init_func=vis.plot_init) 
        plt.show()
        
        target_vis = TargetVisualizer(robot)
        target_sub = rospy.Subscriber('/odom', Odometry, target_vis.new_point)
        cart = FuncAnimation(target_vis.fig, target_vis.update_plot, init_func = target_vis.plot_init)
        plt.show()

        rate.sleep()          

    except rospy.ROSInterruptException:
        print("Action client interrupted", file=sys.stderr)